# Open Model Fabric — Colab model lab

A **standalone** notebook for experimenting with open-weight models on a Google Colab
runtime (CPU or GPU). It:

1. detects the hardware (CPU / GPU / TPU) and shows RAM and VRAM,
2. optionally mounts Google Drive to persist model weights and results,
3. lets you pick a model configuration from a form,
4. downloads weights from the Hugging Face Hub at a **pinnable revision**, safetensors only,
   and records a SHA-256 manifest of every downloaded file,
5. loads the model with `transformers`, runs the platform's `smoke` prompt dataset,
6. measures cold/warm latency, time-to-first-token (TTFT) and tokens/second,
7. saves the experiment as JSON and exports benchmark results as CSV + JSON in the same
   column layout as the Open Model Fabric benchmark export,
8. releases the model's memory.

The notebook does **not** import the Open Model Fabric code base: it only shares file
formats with it, so it runs anywhere `torch` + `transformers` are installed.

## Usage policy — read this first

* **Colab is an interactive notebook environment.** This notebook is for interactive
  experimentation only. It does **not** try to bypass Colab restrictions, keep the runtime
  alive, avoid idle timeouts, auto-reconnect, run unattended, or expose a public web
  endpoint or tunnel (no ngrok, cloudflared, localtunnel or similar). When the runtime
  disconnects, the experiment simply ends; re-run the cells to start again.
* **Serving from Colab.** The Open Model Fabric app can register an inference endpoint
  hosted on Colab **only where your Colab plan and Google's terms explicitly authorize
  it**. That is expressed through the `colab_notebook` compute adapter's
  `endpoint.authorized` attestation in `configs/colab-experiment.yaml`; an endpoint that is
  not marked authorized is recorded but never contacted. Otherwise, serve models with a
  local runtime (Ollama, see `configs/local-ollama.yaml`) or dedicated inference
  infrastructure, and use this notebook only for experiments.
* **Model licenses.** Open-weight model licenses are *not* the same as open-source
  software licenses: each model has its own terms (use restrictions, attribution,
  redistribution). The license noted for each configuration below comes from the model
  card at the time of writing — always re-check the model card for the revision you use.
* **Getting results into the platform.** Exported files land in the output directory
  (on Drive when it is mounted). Upload them to the platform's artifact storage — e.g.
  the Google Drive storage adapter (`storage` id `gdrive` in
  `configs/colab-experiment.yaml`) — through the API, so the platform records size and
  SHA-256 checksum. The last section prints the exact commands.

## 1. Settings

Use the form on the right in Colab (or edit the values). `MODEL_REVISION` defaults to
`main`; **pin a commit hash** for reproducible experiments.

Automated testing: with the environment variable `OMF_NOTEBOOK_SMOKE=1` the notebook
builds a tiny randomly initialised Qwen2 model locally instead of downloading one, and
writes outputs to `OMF_NOTEBOOK_OUTPUT_DIR` (default `./omf_lab_outputs`). Every other
code path is the same. Smoke-mode numbers are **not** meaningful benchmarks.

In [ ]:
# @title Experiment settings { display-mode: "form" }
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # @param ["Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-7B-Instruct"]
# Branch, tag or commit. Pin a full commit hash (see the model's "Files and versions" tab)
# so a re-run downloads exactly the same weights.
MODEL_REVISION = "main"  # @param {type:"string"}
USE_DRIVE = False  # @param {type:"boolean"}
MAX_NEW_TOKENS = 64  # @param {type:"integer"}
TEMPERATURE = 0.0  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}
WARM_ITERATIONS = 2  # @param {type:"integer"}

import os
import sys
from pathlib import Path

SMOKE = os.environ.get("OMF_NOTEBOOK_SMOKE", "").strip().lower() in {"1", "true", "yes"}
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
PROVIDER_ID = "colab-notebook"

if SMOKE:
    MAX_NEW_TOKENS = min(int(MAX_NEW_TOKENS), 16)
    WARM_ITERATIONS = min(int(WARM_ITERATIONS), 2)
    print("=" * 78)
    print("SMOKE MODE (OMF_NOTEBOOK_SMOKE=1): a tiny RANDOM model is built locally.")
    print("No downloads are made. Results are NOT meaningful benchmarks.")
    print("=" * 78)

print(f"In Colab: {IN_COLAB} | smoke mode: {SMOKE}")
print(f"Selected model: {MODEL_ID} @ {MODEL_REVISION}")

In [ ]:
# Dependencies. Colab runtimes ship torch, transformers, huggingface_hub, safetensors and
# psutil; elsewhere install them yourself, e.g. `pip install torch transformers
# huggingface_hub safetensors psutil`.
import importlib.util
import subprocess

REQUIRED = {"torch": "torch", "transformers": "transformers", "safetensors": "safetensors"}
if not SMOKE:
    REQUIRED["huggingface_hub"] = "huggingface_hub"
else:
    REQUIRED["tokenizers"] = "tokenizers"

missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing and IN_COLAB and not SMOKE:
    print("Installing missing packages:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
elif missing:
    raise ImportError(f"Missing packages: {missing}. Install them and re-run this cell.")

import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__)

## 2. Hardware

Detects CPU, CUDA GPUs and (gracefully) the absence of a TPU. `HARDWARE_FINGERPRINT` is
computed exactly like the platform's benchmark service
(`sha256("cpu_model|cpu_count|architecture|accelerator|gpus")[:16]`). Note that the
platform's registered Colab environment uses *declared* resources, so its fingerprint can
differ from the detected one shown here.

In [ ]:
import hashlib
import platform


def cpu_model() -> str | None:
    cpuinfo = Path("/proc/cpuinfo")
    if cpuinfo.exists():
        for line in cpuinfo.read_text(errors="replace").splitlines():
            if line.lower().startswith("model name"):
                return line.split(":", 1)[1].strip()
    return platform.processor() or None


def detect_tpu() -> str | None:
    """Return a description if a TPU looks attached; never raises."""
    for var in ("COLAB_TPU_ADDR", "TPU_NAME", "TPU_ACCELERATOR_TYPE"):
        if os.environ.get(var):
            return f"{var}={os.environ[var]}"
    accel = sorted(str(p) for p in Path("/dev").glob("accel*"))
    if accel:
        return f"TPU device nodes present: {accel}"
    return None


gpus = []
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        gpus.append(
            {
                "index": i,
                "name": props.name,
                "memory_total_bytes": int(props.total_memory),
                "compute_capability": f"{props.major}.{props.minor}",
            }
        )

tpu = detect_tpu()
if gpus:
    ACCELERATOR = "cuda"
elif platform.system() == "Darwin" and platform.machine() == "arm64":
    ACCELERATOR = "mps"
else:
    ACCELERATOR = "none"

HARDWARE = {
    "cpu_model": cpu_model(),
    "cpu_count": os.cpu_count(),
    "architecture": platform.machine(),
    "system": f"{platform.system()} {platform.release()}",
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda_runtime": torch.version.cuda,
    "accelerator": ACCELERATOR,
    "gpus": gpus,
    "tpu": tpu,
    "in_colab": IN_COLAB,
}
_gpu_names = ",".join(sorted(g["name"] for g in gpus)) or "no-gpu"
_raw = (
    f"{HARDWARE['cpu_model']}|{HARDWARE['cpu_count']}|{HARDWARE['architecture']}|"
    f"{ACCELERATOR}|{_gpu_names}"
)
HARDWARE_FINGERPRINT = hashlib.sha256(_raw.encode()).hexdigest()[:16]
HARDWARE["fingerprint"] = HARDWARE_FINGERPRINT

print(f"CPU: {HARDWARE['cpu_model']} x{HARDWARE['cpu_count']} ({HARDWARE['architecture']})")
if gpus:
    for g in gpus:
        print(f"GPU {g['index']}: {g['name']} | {g['memory_total_bytes'] / 2**30:.1f} GiB "
              f"| compute capability {g['compute_capability']}")
else:
    print("GPU: none detected (CPU inference; choose the 0.5B model)")
if tpu:
    print(f"TPU: detected ({tpu}). This lab does not use TPUs; it will run on CPU.")
else:
    print("TPU: not detected (not needed by this notebook)")
print("Hardware fingerprint:", HARDWARE_FINGERPRINT)

## 3. Memory (RAM and VRAM)

In [ ]:
def ram_bytes() -> tuple[int | None, int | None]:
    """(total, available) RAM in bytes via psutil, falling back to /proc/meminfo."""
    try:
        import psutil

        vm = psutil.virtual_memory()
        return int(vm.total), int(vm.available)
    except ImportError:
        pass
    meminfo = Path("/proc/meminfo")
    if meminfo.exists():
        values = {}
        for line in meminfo.read_text().splitlines():
            key, _, rest = line.partition(":")
            parts = rest.split()
            if parts and parts[0].isdigit():
                values[key] = int(parts[0]) * 1024
        return values.get("MemTotal"), values.get("MemAvailable")
    return None, None


def vram_bytes() -> list[dict]:
    out = []
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            free, total = torch.cuda.mem_get_info(i)
            out.append({"index": i, "free_bytes": int(free), "total_bytes": int(total)})
    return out


def show_memory(label: str = "") -> dict:
    total, avail = ram_bytes()
    snap = {"ram_total_bytes": total, "ram_available_bytes": avail, "vram": vram_bytes()}

    def gib(b):
        return f"{b / 2**30:.2f} GiB" if b is not None else "unknown"

    print(f"[{label}] RAM: {gib(avail)} available of {gib(total)}")
    for v in snap["vram"]:
        print(f"[{label}] VRAM GPU {v['index']}: {gib(v['free_bytes'])} free "
              f"of {gib(v['total_bytes'])}")
    if not snap["vram"]:
        print(f"[{label}] VRAM: no CUDA device")
    return snap


HARDWARE["memory_at_start"] = show_memory("start")

## 4. Optional: Google Drive

Mounted only when `USE_DRIVE` is enabled **and** the notebook runs in Colab. Drive keeps
downloaded weights and results across runtime restarts. The `google.colab` import is
deferred into this guard so the notebook also runs outside Colab.

In [ ]:
DRIVE_ROOT = Path("/content/drive")
DRIVE_MOUNTED = False

if USE_DRIVE and IN_COLAB and not SMOKE:
    try:
        from google.colab import drive  # only importable inside Colab

        drive.mount(str(DRIVE_ROOT))
        DRIVE_MOUNTED = (DRIVE_ROOT / "MyDrive").exists()
    except Exception as exc:  # user declined, quota, not in Colab...
        print(f"Drive not mounted ({type(exc).__name__}: {exc}); using local storage.")
elif USE_DRIVE:
    print("USE_DRIVE is set but this is not a Colab runtime (or smoke mode); skipping.")

if DRIVE_MOUNTED:
    BASE_DIR = DRIVE_ROOT / "MyDrive" / "open-model-fabric"
    MODEL_CACHE_DIR = BASE_DIR / "models"
    default_output = BASE_DIR / "experiments"
else:
    default_cache = "/content/omf-models" if IN_COLAB else "~/.cache/open-model-fabric/models"
    MODEL_CACHE_DIR = Path(os.environ.get("OMF_MODEL_CACHE", default_cache)).expanduser()
    default_output = Path("omf_lab_outputs")

OUTPUT_DIR = Path(os.environ.get("OMF_NOTEBOOK_OUTPUT_DIR") or default_output).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not SMOKE:
    MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Drive mounted:", DRIVE_MOUNTED)
print("Model cache:  ", MODEL_CACHE_DIR if not SMOKE else "(smoke: temporary directory)")
print("Outputs:      ", OUTPUT_DIR.resolve())

## 5. Model configurations

| Model | Params | Runs on | License (model card) |
|---|---|---|---|
| `Qwen/Qwen2.5-0.5B-Instruct` (default) | 0.5B | CPU or GPU | Apache-2.0 |
| `Qwen/Qwen2.5-1.5B-Instruct` | 1.5B | CPU (slow) or GPU | Apache-2.0 |
| `Qwen/Qwen2.5-7B-Instruct` | 7.6B | GPU only, ~16 GB VRAM in fp16/bf16 | Apache-2.0 |

The licenses above are what the model cards state; open-weight licenses are model
licenses, not open-source *software* licenses, and can change between revisions.
Verify the card for the revision you pin. The 7B model does not fit a 15 GB T4 in fp16
without quantization; the notebook warns before loading it.

In [ ]:
MODEL_CONFIGS = {
    "Qwen/Qwen2.5-0.5B-Instruct": {
        "params_b": 0.49,
        "license": "Apache-2.0",
        "license_url": "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/blob/main/LICENSE",
        "gpu_only": False,
        "min_vram_gb": 2,
    },
    "Qwen/Qwen2.5-1.5B-Instruct": {
        "params_b": 1.54,
        "license": "Apache-2.0",
        "license_url": "https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/blob/main/LICENSE",
        "gpu_only": False,
        "min_vram_gb": 5,
    },
    "Qwen/Qwen2.5-7B-Instruct": {
        "params_b": 7.62,
        "license": "Apache-2.0",
        "license_url": "https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/blob/main/LICENSE",
        "gpu_only": True,
        "min_vram_gb": 17,
    },
}

if MODEL_ID not in MODEL_CONFIGS:
    raise ValueError(f"Unknown model {MODEL_ID!r}; choose one of {list(MODEL_CONFIGS)}")
MODEL_CONFIG = dict(MODEL_CONFIGS[MODEL_ID])

# Device and precision: bf16 on GPUs that support it, else fp16; fp32 on CPU.
if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    DEVICE = "cpu"
    DTYPE = torch.float32
DTYPE_NAME = str(DTYPE).replace("torch.", "")
QUANTIZATION = "none"  # weights are loaded unquantized in DTYPE

if not SMOKE:
    if MODEL_CONFIG["gpu_only"] and DEVICE != "cuda":
        raise RuntimeError(
            f"{MODEL_ID} is configured as GPU-only. Switch the runtime to a GPU "
            "(Runtime > Change runtime type) or pick the 0.5B model."
        )
    if DEVICE == "cuda":
        free_gb = torch.cuda.mem_get_info(0)[0] / 2**30
        if free_gb < MODEL_CONFIG["min_vram_gb"]:
            print(
                f"WARNING: {MODEL_ID} needs roughly {MODEL_CONFIG['min_vram_gb']} GB of VRAM "
                f"in {DTYPE_NAME}, but only {free_gb:.1f} GB is free. Loading will likely "
                "fail with CUDA out-of-memory; choose a smaller model or a larger GPU."
            )

print(f"Model: {MODEL_ID} | license: {MODEL_CONFIG['license']} (verify on the model card)")
print(f"Device: {DEVICE} | dtype: {DTYPE_NAME} | quantization: {QUANTIZATION}")

## 6. Download weights (trusted source, pinned revision, safetensors only)

Weights come from the Hugging Face Hub via `huggingface_hub.snapshot_download`:

* the requested revision is first resolved to an exact **commit hash**, which is what is
  downloaded and recorded (pin `MODEL_REVISION` to that hash to reproduce a run);
* `allow_patterns` restricts the download to safetensors weights, JSON configs, tokenizer
  files and the license — pickle-based `*.bin`/`*.pt` weights are never downloaded;
* remote code is never executed (`trust_remote_code` is left at its default, off).

In smoke mode a tiny random Qwen2 model and a small `tokenizers`-based tokenizer are built
in a temporary directory instead; everything after this cell is identical.

In [ ]:
import tempfile

ALLOW_PATTERNS = [
    "*.safetensors",
    "*.safetensors.index.json",
    "*.json",  # config.json, generation_config.json, tokenizer.json, tokenizer_config.json
    "merges.txt",
    "vocab.*",
    "tokenizer*",
    "*.jinja",  # chat templates stored as separate files by newer tokenizers
    "LICENSE*",
]
FORBIDDEN_SUFFIXES = (".bin", ".pt", ".pth", ".pkl", ".pickle", ".ckpt")


def download_from_hub(repo_id: str, revision: str, cache_root: Path) -> tuple[Path, str]:
    from huggingface_hub import HfApi, snapshot_download

    info = HfApi().model_info(repo_id, revision=revision)
    resolved = info.sha
    siblings = [s.rfilename for s in (info.siblings or [])]
    if not any(name.endswith(".safetensors") for name in siblings):
        raise RuntimeError(f"{repo_id}@{resolved} publishes no safetensors weights; refusing.")
    card_license = getattr(info.card_data, "license", None) if info.card_data else None
    if card_license:
        print(f"Model card license tag: {card_license}")
        MODEL_CONFIG["license_from_card"] = card_license
    local_dir = cache_root / repo_id.replace("/", "__") / resolved
    path = snapshot_download(
        repo_id=repo_id,
        revision=resolved,  # exact commit, so the cache directory matches the content
        allow_patterns=ALLOW_PATTERNS,
        local_dir=str(local_dir),
    )
    return Path(path), resolved


def build_smoke_model(seed: int) -> tuple[Path, str]:
    """Tiny random Qwen2 + word-level tokenizer, built offline. NOT a real model."""
    from tokenizers import Tokenizer, models, pre_tokenizers, trainers
    from transformers import PreTrainedTokenizerFast, Qwen2Config, Qwen2ForCausalLM

    specials = ["<|endoftext|>", "<|im_start|>", "<|im_end|>", "[UNK]"]
    corpus = [
        "system user assistant",
        "Say hello in one short sentence.",
        "Count from one to ten, separated by commas.",
        "one two three four five six seven eight nine ten",
        "Define 'latency' in two sentences.",
        "Hello! Latency is the time between a request and its response.",
        "You are a helpful assistant. The quick brown fox jumps over the lazy dog.",
    ]
    tok = Tokenizer(models.WordLevel(unk_token="[UNK]"))
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    tok.train_from_iterator(corpus, trainers.WordLevelTrainer(special_tokens=specials))
    tok.add_special_tokens(specials)
    tokenizer_fast = PreTrainedTokenizerFast(
        tokenizer_object=tok,
        unk_token="[UNK]",
        eos_token="<|im_end|>",
        pad_token="<|endoftext|>",
    )
    tokenizer_fast.chat_template = (
        "{% for message in messages %}<|im_start|>{{ message['role'] }}\n"
        "{{ message['content'] }}<|im_end|>\n{% endfor %}"
        "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}"
    )
    torch.manual_seed(seed)
    config = Qwen2Config(
        vocab_size=len(tokenizer_fast),
        hidden_size=64,
        intermediate_size=128,
        num_hidden_layers=2,
        num_attention_heads=4,
        num_key_value_heads=2,
        max_position_embeddings=256,
        tie_word_embeddings=True,
        bos_token_id=None,
        eos_token_id=tokenizer_fast.eos_token_id,
        pad_token_id=tokenizer_fast.pad_token_id,
    )
    out = Path(tempfile.mkdtemp(prefix="omf-smoke-model-"))
    Qwen2ForCausalLM(config).save_pretrained(out)  # safetensors by default
    tokenizer_fast.save_pretrained(out)
    return out, "smoke-local"


if SMOKE:
    SOURCE = "smoke-local-random"
    MODEL_ID = "omf-smoke/tiny-random-qwen2"
    MODEL_CONFIG = {"params_b": 0.0, "license": "n/a (randomly initialised, smoke test)",
                    "gpu_only": False, "min_vram_gb": 0}
    MODEL_DIR, RESOLVED_REVISION = build_smoke_model(SEED)
else:
    SOURCE = "huggingface-hub"
    MODEL_DIR, RESOLVED_REVISION = download_from_hub(MODEL_ID, MODEL_REVISION, MODEL_CACHE_DIR)

print(f"Weights for {MODEL_ID} @ {RESOLVED_REVISION} in {MODEL_DIR}")

## 7. Checksum manifest

Every model file gets its size and SHA-256 recorded in `omf-manifest.json` (next to the
weights and copied to the output directory), mirroring the platform's artifact-storage
checksum model. On a re-run with cached weights, the files are verified against the
previous manifest before they are used.

In [ ]:
import json
import shutil
from datetime import UTC, datetime

MANIFEST_NAME = "omf-manifest.json"


def sha256_file(path: Path, chunk: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        while block := fh.read(chunk):
            digest.update(block)
    return digest.hexdigest()


def model_files(root: Path) -> list[Path]:
    return sorted(
        p for p in root.rglob("*")
        if p.is_file() and ".cache" not in p.relative_to(root).parts and p.name != MANIFEST_NAME
    )


files = model_files(MODEL_DIR)
pickles = [p.name for p in files if p.suffix.lower() in FORBIDDEN_SUFFIXES]
if pickles:
    raise RuntimeError(f"Refusing to use pickle-based files: {pickles}")
if not any(p.suffix == ".safetensors" for p in files):
    raise RuntimeError("No safetensors weights found")

entries = [
    {"path": p.relative_to(MODEL_DIR).as_posix(), "size_bytes": p.stat().st_size,
     "sha256": sha256_file(p)}
    for p in files
]
manifest_path = MODEL_DIR / MANIFEST_NAME
if manifest_path.exists():
    previous = {e["path"]: e["sha256"] for e in json.loads(manifest_path.read_text())["files"]}
    changed = [e["path"] for e in entries if e["path"] in previous
               and previous[e["path"]] != e["sha256"]]
    if changed:
        raise RuntimeError(f"Cached files changed since they were recorded: {changed}")
    print(f"Verified {len(previous)} cached file(s) against the previous manifest.")

MANIFEST = {
    "format": "omf.model-manifest.v1",
    "model_id": MODEL_ID,
    "requested_revision": MODEL_REVISION if not SMOKE else "n/a",
    "resolved_revision": RESOLVED_REVISION,
    "source": SOURCE,
    "license": MODEL_CONFIG.get("license"),
    "created_at": datetime.now(UTC).isoformat(),
    "total_size_bytes": sum(e["size_bytes"] for e in entries),
    "files": entries,
}
manifest_path.write_text(json.dumps(MANIFEST, indent=2))
for e in entries:
    print(f"{e['sha256'][:16]}  {e['size_bytes']:>12,d}  {e['path']}")
print(f"{len(entries)} files, {MANIFEST['total_size_bytes'] / 2**20:.1f} MiB")

## 8. Load the model

`AutoTokenizer` / `AutoModelForCausalLM` from the local directory only, safetensors only,
no remote code. Precision: bfloat16/float16 on GPU, float32 on CPU.

In [ ]:
import time

from packaging.version import Version
from transformers import AutoModelForCausalLM, AutoTokenizer

# transformers >= 4.56 renamed `torch_dtype` to `dtype`.
_dtype_kw = "dtype" if Version(transformers.__version__) >= Version("4.56") else "torch_dtype"
load_kwargs = {_dtype_kw: DTYPE, "use_safetensors": True}
if DEVICE == "cuda" and importlib.util.find_spec("accelerate") is not None:
    load_kwargs["device_map"] = {"": 0}  # stream weights straight to the GPU

_t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), **load_kwargs)
if "device_map" not in load_kwargs:
    model = model.to(DEVICE)
model.eval()
LOAD_SECONDS = round(time.perf_counter() - _t0, 3)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Loaded {MODEL_ID} in {LOAD_SECONDS}s on {model.device} ({model.dtype}); "
      f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")
HARDWARE["memory_after_load"] = show_memory("after load")

## 9. Sample prompts

The platform's built-in `smoke` dataset (version `1.0.0`). Keep ids and texts identical so
results stay comparable with platform benchmark runs on the same dataset version.

In [ ]:
DATASET_ID = "smoke"
DATASET_VERSION = "1.0.0"
PROMPTS = [
    {"id": "greet", "text": "Say hello in one short sentence."},
    {"id": "count", "text": "Count from one to ten, separated by commas."},
    {"id": "define", "text": "Define 'latency' in two sentences."},
]

GENERATION = {
    "max_new_tokens": int(MAX_NEW_TOKENS),
    "temperature": float(TEMPERATURE),
    "seed": int(SEED),
    "do_sample": float(TEMPERATURE) > 0,
}
if SMOKE:
    # A random model may emit EOS immediately; force a fixed length so throughput is
    # measurable in the automated test.
    GENERATION["min_new_tokens"] = GENERATION["max_new_tokens"]
print(json.dumps(GENERATION))

## 10. Measure latency, TTFT and tokens/second

`generate()` runs in a background thread and streams through a
`transformers.TextIteratorStreamer`. The streamer subclass timestamps the first generated
token (TTFT), so the measurement is not delayed by the streamer's word buffering.

* **cold**: the first request after loading (prompt `greet`, iteration 0) — it includes
  one-off costs such as CUDA kernel selection and allocator warm-up;
* **warm**: `WARM_ITERATIONS` passes over all prompts afterwards.

`tokens_per_second` uses the platform definition: output tokens / (total − TTFT), i.e. the
decode rate excluding time-to-first-token. `e2e_tokens_per_second` = output tokens / total.
Output tokens are counted exactly with the model's tokenizer.

In [ ]:
import gc
import threading
import uuid

from transformers import TextIteratorStreamer


class TimingStreamer(TextIteratorStreamer):
    """TextIteratorStreamer that also records when the first new token arrives."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.first_token_at = None
        self.streamed_tokens = 0

    def put(self, value):
        if not (self.skip_prompt and self.next_tokens_are_prompt):
            if self.first_token_at is None:
                self.first_token_at = time.perf_counter()
            self.streamed_tokens += int(value.numel())
        super().put(value)


def chat_inputs(prompt_text: str):
    messages = [{"role": "user", "content": prompt_text}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)


def measure(prompt: dict, iteration: int, phase: str) -> dict:
    sample = {
        "provider_id": PROVIDER_ID,
        "model": MODEL_ID,
        "prompt_id": prompt["id"],
        "iteration": iteration,
        "phase": phase,
        "warm_up_status": "cold: first request after model load" if phase == "cold"
        else "warm: model already exercised",
        "ttft_ms": None,
        "total_ms": None,
        "output_tokens": None,
        "token_count_source": None,
        "tokens_per_second": None,
        "e2e_tokens_per_second": None,
        "output_text": None,
        "error_code": None,
        "error_message": None,
    }
    result, failure = {}, {}
    try:
        transformers.set_seed(GENERATION["seed"])
        inputs = chat_inputs(prompt["text"])
        streamer = TimingStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True,
                                  timeout=600)
        gen_kwargs = {
            **inputs,
            "streamer": streamer,
            "max_new_tokens": GENERATION["max_new_tokens"],
            "do_sample": GENERATION["do_sample"],
            "pad_token_id": tokenizer.pad_token_id,
        }
        if GENERATION["do_sample"]:
            gen_kwargs["temperature"] = GENERATION["temperature"]
        else:  # silence "temperature/top_p ignored" warnings from model generation configs
            gen_kwargs.update(temperature=None, top_p=None, top_k=None)
        if "min_new_tokens" in GENERATION:
            gen_kwargs["min_new_tokens"] = GENERATION["min_new_tokens"]

        def _run():
            try:
                with torch.inference_mode():
                    result["ids"] = model.generate(**gen_kwargs)
            except Exception as exc:  # surfaced in the main thread below
                failure["exc"] = exc
                streamer.end()

        if DEVICE == "cuda":
            torch.cuda.synchronize()
        started = time.perf_counter()
        worker = threading.Thread(target=_run, daemon=True)
        worker.start()
        text = "".join(chunk for chunk in streamer)
        worker.join()
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        finished = time.perf_counter()
        if "exc" in failure:
            raise failure["exc"]

        prompt_len = inputs["input_ids"].shape[-1]
        output_tokens = int(result["ids"].shape[-1] - prompt_len)
        total_ms = (finished - started) * 1000
        ttft_ms = ((streamer.first_token_at - started) * 1000
                   if streamer.first_token_at is not None else None)
        decode_ms = total_ms - (ttft_ms or 0.0)
        if decode_ms <= 0:
            decode_ms = total_ms
        sample.update(
            ttft_ms=round(ttft_ms, 2) if ttft_ms is not None else None,
            total_ms=round(total_ms, 2),
            output_tokens=output_tokens,
            token_count_source="provider",  # exact count from the model's own tokenizer
            tokens_per_second=round(output_tokens / (decode_ms / 1000), 2)
            if output_tokens else None,
            e2e_tokens_per_second=round(output_tokens / (total_ms / 1000), 2)
            if output_tokens else None,
            output_text=text,
        )
    except torch.cuda.OutOfMemoryError as exc:
        sample.update(error_code="out_of_memory", error_message=str(exc)[:500])
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as exc:
        sample.update(error_code=type(exc).__name__, error_message=str(exc)[:500])
    return sample


RUN_ID = uuid.uuid4().hex
RUN_STARTED = datetime.now(UTC)
RUN_NAME = (("SMOKE-" if SMOKE else "") + "colab-lab-" + MODEL_ID.split("/")[-1]
            + "-" + RUN_STARTED.strftime("%Y%m%dT%H%M%SZ"))

SAMPLES = [measure(PROMPTS[0], 0, "cold")]
for it in range(int(WARM_ITERATIONS)):
    for p in PROMPTS:
        SAMPLES.append(measure(p, it, "warm"))

for s in SAMPLES:
    status = s["error_code"] or "ok"
    print(f"{s['phase']:<5} {s['prompt_id']:<7} it={s['iteration']} {status:<6} "
          f"ttft={s['ttft_ms']}ms total={s['total_ms']}ms tokens={s['output_tokens']} "
          f"tok/s={s['tokens_per_second']}")
print("\nSample output:", repr((SAMPLES[0]["output_text"] or "")[:200]))
if SMOKE:
    print("SMOKE MODE: random weights; outputs are gibberish and timings are NOT benchmarks.")

## 11. Save the experiment and export benchmark results

* `experiment_<run_id>.json` — full record: model id + requested and resolved revision,
  license, hardware metadata, dtype/quantization, dataset id + version, generation
  settings (including the seed), every sample with warm-up status, outputs and errors,
  and a summary.
* `benchmark_<run_id>.csv` / `benchmark_<run_id>.json` — one row per sample using the
  platform's benchmark export columns (`provider_id` = `colab-notebook`).
* `model-manifest_<run_id>.json` — the checksum manifest from step 7.

In [ ]:
import csv
import statistics

BENCHMARK_COLUMNS = [
    "run_id", "run_name", "dataset_id", "dataset_version", "provider_id", "model",
    "prompt_id", "iteration", "phase", "ttft_ms", "total_ms", "output_tokens",
    "token_count_source", "tokens_per_second", "error_code", "error_message",
    "temperature", "max_tokens", "seed", "quantization", "model_revision",
    "hardware_fingerprint",
]


def percentile(values, pct):
    if not values:
        return None
    ordered = sorted(values)
    k = (len(ordered) - 1) * pct
    lo, hi = int(k), min(int(k) + 1, len(ordered) - 1)
    return round(ordered[lo] + (ordered[hi] - ordered[lo]) * (k - lo), 2)


ok = [s for s in SAMPLES if not s["error_code"]]
cold = next((s for s in ok if s["phase"] == "cold"), None)
warm = [s for s in ok if s["phase"] == "warm"]
SUMMARY = {
    "samples": len(SAMPLES),
    "errors": len(SAMPLES) - len(ok),
    "cold_ttft_ms": cold["ttft_ms"] if cold else None,
    "cold_total_ms": cold["total_ms"] if cold else None,
    "warm_p50_total_ms": percentile([s["total_ms"] for s in warm], 0.5),
    "warm_p95_total_ms": percentile([s["total_ms"] for s in warm], 0.95),
    "warm_mean_ttft_ms": round(statistics.fmean([s["ttft_ms"] for s in warm
                                                 if s["ttft_ms"] is not None]), 2)
    if any(s["ttft_ms"] is not None for s in warm) else None,
    "warm_mean_tokens_per_second": round(statistics.fmean(
        [s["tokens_per_second"] for s in warm if s["tokens_per_second"]]), 2)
    if any(s["tokens_per_second"] for s in warm) else None,
}

rows = [
    {
        **{k: s.get(k) for k in BENCHMARK_COLUMNS if k in s},
        "run_id": RUN_ID,
        "run_name": RUN_NAME,
        "dataset_id": DATASET_ID,
        "dataset_version": DATASET_VERSION,
        "temperature": GENERATION["temperature"],
        "max_tokens": GENERATION["max_new_tokens"],
        "seed": GENERATION["seed"],
        "quantization": QUANTIZATION,
        "model_revision": RESOLVED_REVISION,
        "hardware_fingerprint": HARDWARE_FINGERPRINT,
    }
    for s in SAMPLES
]

NOTES = [
    "'cold' is the first request after loading the model in this runtime.",
    "tokens_per_second = output_tokens / (total_ms - ttft_ms); "
    "e2e_tokens_per_second = output_tokens / total_ms.",
    "Colab hardware varies between sessions; compare runs only with equal "
    "hardware_fingerprint, dtype, quantization, dataset version and settings.",
]
if SMOKE:
    NOTES.insert(0, "SMOKE MODE: tiny randomly initialised model; NOT a meaningful benchmark.")

EXPERIMENT = {
    "format": "omf.notebook-experiment.v1",
    "smoke": SMOKE,
    "run_id": RUN_ID,
    "run_name": RUN_NAME,
    "started_at": RUN_STARTED.isoformat(),
    "completed_at": datetime.now(UTC).isoformat(),
    "provider_id": PROVIDER_ID,
    "model": {
        "id": MODEL_ID,
        "requested_revision": MODEL_REVISION if not SMOKE else "n/a",
        "revision": RESOLVED_REVISION,
        "source": SOURCE,
        "license": MODEL_CONFIG.get("license"),
        "license_from_card": MODEL_CONFIG.get("license_from_card"),
        "dtype": DTYPE_NAME,
        "quantization": QUANTIZATION,
        "device": DEVICE,
        "load_seconds": LOAD_SECONDS,
        "manifest_sha256": hashlib.sha256(
            json.dumps(MANIFEST["files"], sort_keys=True).encode()).hexdigest(),
    },
    "hardware": HARDWARE,
    "dataset": {"id": DATASET_ID, "version": DATASET_VERSION, "prompts": PROMPTS},
    "generation": GENERATION,
    "warm_iterations": int(WARM_ITERATIONS),
    "samples": SAMPLES,
    "summary": SUMMARY,
    "notes": NOTES,
}

experiment_path = OUTPUT_DIR / f"experiment_{RUN_ID}.json"
csv_path = OUTPUT_DIR / f"benchmark_{RUN_ID}.csv"
bench_json_path = OUTPUT_DIR / f"benchmark_{RUN_ID}.json"
manifest_copy = OUTPUT_DIR / f"model-manifest_{RUN_ID}.json"

experiment_path.write_text(json.dumps(EXPERIMENT, indent=2, default=str))
with csv_path.open("w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=BENCHMARK_COLUMNS, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)
bench_json_path.write_text(json.dumps({
    "format": "omf.notebook-benchmark.v1",
    "smoke": SMOKE,
    "columns": BENCHMARK_COLUMNS,
    "summary": SUMMARY,
    "notes": NOTES,
    "rows": rows,
}, indent=2))
shutil.copyfile(manifest_path, manifest_copy)

EXPORTS = [experiment_path, csv_path, bench_json_path, manifest_copy]
for path in EXPORTS:
    print(f"wrote {path} ({path.stat().st_size:,d} bytes, sha256 {sha256_file(path)[:16]})")
print(json.dumps(SUMMARY, indent=2))
if SMOKE:
    print("SMOKE MODE: these files exercise the export path only; they are NOT benchmarks.")

## 12. Upload results to the platform

The notebook never opens network listeners or tunnels and never calls the platform for you.
To store results in the platform's artifact storage:

1. Keep the files: with Drive mounted they are already in
   `MyDrive/open-model-fabric/experiments`; otherwise download them from the Colab file
   browser.
2. From a machine that can reach your Open Model Fabric API, upload each file to a storage
   provider — e.g. the Google Drive storage adapter (`gdrive` in
   `configs/colab-experiment.yaml`) or `local`. Uploading through the API (rather than
   copying files into the Drive folder) lets the platform validate the key and content
   type and record size and SHA-256. Compare the printed checksums with the ones above.

In [ ]:
STORAGE_ID = "gdrive"  # or "local"; any storage id configured in your profile
CONTENT_TYPES = {".csv": "text/csv", ".json": "application/json"}
print("# Run these from a machine that can reach your Open Model Fabric API:")
print('export OMF_URL="http://127.0.0.1:8080"   # your API base URL')
print('export OMF_TOKEN="..."                     # admin token, if auth is enabled')
for path in EXPORTS:
    key = f"experiments/colab/{RUN_ID}/{path.name}"
    print(
        f'curl -fsS -X POST "$OMF_URL/api/v1/storage/{STORAGE_ID}/artifacts" '
        f'-H "Authorization: Bearer $OMF_TOKEN" -F "key={key}" '
        f'-F "file=@{path.name};type={CONTENT_TYPES[path.suffix]}"'
    )

## 13. Release resources

Frees the model and tokenizer and returns cached GPU memory, so you can load another
configuration in the same session — or simply disconnect the runtime when you are done
(Runtime > Disconnect and delete runtime), which is the polite thing to do on shared
infrastructure.

In [ ]:
for _name in ("model", "tokenizer", "streamer", "inputs"):
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
HARDWARE["memory_after_cleanup"] = show_memory("after cleanup")
if SMOKE:
    shutil.rmtree(MODEL_DIR, ignore_errors=True)
print("Model resources released.")